# 🕷️ Spidey Tracker — K-NN Classification & K-Means Clustering

**Dataset:** [Spidey Tracker: Spiderman Dataset](https://www.kaggle.com/datasets/umuttuygurr/spidey-tracker-spiderman-dataset)

**Goal:** Predict whether a Spider-Man sighting is verified or not using K-NN, and discover sighting clusters using K-Means.

This notebook demonstrates progressive model improvement — each optimization step is measured and compared.

## 1. Setup & Imports

In [ ]:
# Install kagglehub to download the dataset
!pip install -q kagglehub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Download & Load Dataset

In [ ]:
import kagglehub

# Download dataset from Kaggle
path = kagglehub.dataset_download("umuttuygurr/spidey-tracker-spiderman-dataset")
print("Dataset downloaded to:", path)

# List all files in the dataset
for root, dirs, files in os.walk(path):
    for file in files:
        filepath = os.path.join(root, file)
        print(f"  {file} ({os.path.getsize(filepath) / 1024:.0f} KB)")

In [ ]:
# Load the main sightings dataset
df = pd.read_csv(os.path.join(path, 'sightings.csv'))
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info about the dataset
print("=== Column Types ===")
print(df.dtypes)
print(f"\n=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Statistical summary of numeric columns
df.describe()

In [ ]:
# Target variable distribution
print("=== Target: is_verified ===")
print(df['is_verified'].value_counts())
print(f"\nVerified ratio: {df['is_verified'].mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['is_verified'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Verified vs Unverified Sightings')
axes[0].set_xlabel('Is Verified (0=No, 1=Yes)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df['is_verified'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                       colors=['#e74c3c', '#2ecc71'], labels=['Not Verified', 'Verified'])
axes[1].set_title('Sighting Verification Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")

plt.figure(figsize=(14, 10))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key numeric features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Plot first 6 numeric features (skip IDs)
plot_cols = [c for c in numeric_cols if 'id' not in c.lower()][:6]
for i, col in enumerate(plot_cols):
    axes[i].hist(df[col], bins=50, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of Key Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

We need to:
1. Select relevant numeric features (drop IDs, targets, metadata)
2. Handle missing values
3. Split into train/test sets

In [ ]:
# Define target
target = 'is_verified'
y = df[target]

# Select feature columns: all numeric columns EXCEPT targets, IDs, and metadata
drop_keywords = ['id', 'target', 'verified', 'verification', 'split']
feature_cols = [c for c in numeric_cols
                if not any(kw in c.lower() for kw in drop_keywords)]

print(f"Selected {len(feature_cols)} features:")
for col in feature_cols:
    print(f"  - {col}")

X = df[feature_cols].copy()

In [ ]:
# Handle missing values (fill with median)
missing_before = X.isnull().sum().sum()
X = X.fillna(X.median())
print(f"Missing values filled: {missing_before} -> {X.isnull().sum().sum()}")

In [ ]:
# Train/test split — use the dataset's built-in split column if available
if 'split' in df.columns:
    train_mask = df['split'] == 'train'
    test_mask = df['split'] == 'test'
    X_train, X_test = X[train_mask].values, X[test_mask].values
    y_train, y_test = y[train_mask].values, y[test_mask].values
    print("Using dataset's built-in split column")
else:
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X.values, y.values, test_size=0.2, random_state=42, stratify=y)
    print("Using 80/20 stratified random split")

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Train verified ratio: {y_train.mean():.2%}")
print(f"Test verified ratio:  {y_test.mean():.2%}")

## 5. K-NN Classification

### Step 1: Baseline Model (Default Parameters)

We start with scikit-learn's default KNN: K=5, uniform weights, Minkowski distance.
No scaling, no tuning — just the raw baseline.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# We'll track every improvement step here
results = []

# Baseline: default KNN, no scaling
knn_baseline = KNeighborsClassifier()  # k=5, uniform weights, minkowski
knn_baseline.fit(X_train, y_train)
y_pred_baseline = knn_baseline.predict(X_test)

acc_baseline = accuracy_score(y_test, y_pred_baseline)
results.append({'Step': '1. Baseline (defaults, no scaling)', 'Accuracy': acc_baseline, 'Details': 'K=5, uniform, minkowski'})

print(f"Baseline Accuracy: {acc_baseline:.4f}")
print(f"\n{classification_report(y_test, y_pred_baseline)}")

In [ ]:
# Confusion matrix for baseline
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_baseline)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Verified', 'Verified'],
            yticklabels=['Not Verified', 'Verified'])
ax.set_title(f'Baseline KNN — Accuracy: {acc_baseline:.4f}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()